# Setup and IResNet-100 smoke test

This stage validates the persisted environment and prepared data before candidate training. It intentionally stops on any contract violation.


In [12]:
# Configuration
from pathlib import Path
import sys

REPO_ROOT = next(
    root for root in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (root / "LICENSE.md").exists() and (root / "face_name_metric_learning").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
MANIFEST_PATH = REPO_ROOT / "data/processed/manifest.csv"
HDF5_PATH = REPO_ROOT / "data/processed/faces.h5"
ENVIRONMENT_RECORD_PATH = REPO_ROOT / "artifacts/face_name_metric_learning/setup_environment.json"
PRETRAINED_WEIGHTS_PATH = REPO_ROOT / "models/pretrained/iresnet100.pth"
PRETRAINED_MODULE = "face_name_metric_learning.iresnet"
PRETRAINED_WEIGHT_IDENTIFIER = "local-converted:iresnet100"
PRETRAINED_WEIGHT_SHA256 = "8a51c4a5607d4d6db1e160562a01721b7b6516645d62325be47e36cf15eb9e59"
EXPECTED_SPLIT_COUNTS = {"train": 45078, "validation": 9660, "test": 9660}
EXPECTED_CLASS_COUNT = 100
EMBEDDING_DIMENSION = 512
PINNED_TRAINING_PACKAGES = ("torch==2.8.0", "pytorch-metric-learning==2.8.1", "h5py==3.16.0")


## Install and import the pinned runtime
Before continuing, install any reported requirements with the displayed `%pip` command.


In [13]:
from importlib import metadata
missing_or_wrong = []
for requirement in PINNED_TRAINING_PACKAGES:
    name, expected = requirement.split("==", 1)
    try: actual = metadata.version(name)
    except metadata.PackageNotFoundError: actual = None
    if actual is None or actual.split("+", 1)[0] != expected: missing_or_wrong.append(requirement)
if missing_or_wrong:
    command = "install " + " ".join(missing_or_wrong)
    get_ipython().run_line_magic("pip", command)
    raise RuntimeError("Restart the kernel after installing the pinned runtime")


In [14]:
import h5py
import torch
from face_name_metric_learning import collect_environment_record, sha256_file, validate_manifest_hdf5, write_environment_record
from face_name_metric_learning.iresnet import load_iresnet100
assert REPO_ROOT.is_dir()
assert MANIFEST_PATH.is_file(), f"Missing manifest: {MANIFEST_PATH}"
assert HDF5_PATH.is_file(), f"Missing prepared faces: {HDF5_PATH}"
data_record = validate_manifest_hdf5(MANIFEST_PATH, HDF5_PATH, expected_split_counts=EXPECTED_SPLIT_COUNTS, expected_class_count=EXPECTED_CLASS_COUNT)
print(data_record)


AssertionError: Missing prepared faces: /nfs/home/aqu4eb/DS4002_Project3/data/processed/faces.h5

## Load one pretrained image encoder and smoke-test one known HDF5 row
No split is augmented. A 512-dimensional row is normalized in float32 before the assertion.


In [16]:
assert PRETRAINED_WEIGHTS_PATH.is_file(), f"Missing pretrained weights: {PRETRAINED_WEIGHTS_PATH}"
weight_sha256 = sha256_file(PRETRAINED_WEIGHTS_PATH)
if PRETRAINED_WEIGHT_SHA256 is not None: assert weight_sha256 == PRETRAINED_WEIGHT_SHA256
environment_record = collect_environment_record()
model = load_iresnet100(PRETRAINED_WEIGHTS_PATH, environment_record["device"])
model.eval()
with h5py.File(HDF5_PATH, "r") as h5:
    known_index = int(h5["split_indices/validation"][0])
    image = torch.from_numpy(h5["images"][known_index:known_index + 1]).to(environment_record["device"])
with torch.inference_mode():
    embedding = model(image).detach().to(dtype=torch.float32, device="cpu").squeeze(0)
embedding = torch.nn.functional.normalize(embedding, p=2, dim=0)
assert embedding.shape == (EMBEDDING_DIMENSION,), embedding.shape
assert torch.isfinite(embedding).all()
assert torch.isclose(torch.linalg.vector_norm(embedding), torch.tensor(1.0), atol=1e-5)


UnpicklableState: some of this cell's state could not be saved:
  - h5: TypeError: h5py objects cannot be pickled
These names stay usable on this worker only; a cell that needs them on another worker (or after a restart) will fail with this reason.


In [17]:
environment_record.update({"data": data_record, "paths": {"manifest": str(MANIFEST_PATH), "faces_hdf5": str(HDF5_PATH)}, "pretrained_weights": {"identifier": PRETRAINED_WEIGHT_IDENTIFIER, "sha256": weight_sha256}, "smoke_test": {"h5_index": known_index, "embedding_shape": list(embedding.shape), "embedding_dtype": str(embedding.dtype), "normalized": True}})
write_environment_record(ENVIRONMENT_RECORD_PATH, environment_record)
print(ENVIRONMENT_RECORD_PATH)


/nfs/home/aqu4eb/DS4002_Project3/artifacts/face_name_metric_learning/setup_environment.json
